# Introduction to Docker Compose

Here, I will build a simple ETL (Extract, Transform, Load) workflow using Docker Compose including two services 1) PostgreSQL to store data and 2) Python to load and process the data. This setup mirrors how real-world data pipelines are often prototyped and tested because Compose gives you a reliable, repeatable way to build and share these workflows

At the end of the section, I will be able to:

1. Build and run custom Docker images
2. Define multi-service environments with a Compose file
3. Pass environment variables and connect services
4. Use volumes for persistent storage
5. Run, inspect, and reuse your full stack with one command

## What is Docker Compose?

I have being working with `docker run` command in which Docker runs one container at a time. This method work greats for tests or small project, but the error rates when copy/paste the information is high and it will increase when we need multiple services instead of one.

So what can we do to avoid errors and simplify the process? We can use Docker Compose to simplify the process!

Docker Compose simplifies this by letting you define your setup in a single file named `docker-compose.yaml`. The file describes:
1. each service in your app
2. how they connect
3. how to configure the services. 

Once the `docker-compose.yaml` is finished and placed, Compose can proceed to build images, start containers in the correct order, and connect everything over a shared network, all in one step.

To see how that works in practice, we’ll start by launching a PostgreSQL database with Docker Compose for our project of building a simple ETL. From there, we’ll add a second container that runs a Python script and connects to the database.

## Running Postgres with Docker Compose

In [4]:
# Create Docker Compose Demonstration directory
mkdir compose-demo
cd compose-demo

# Write Docker Compose file 
vim docker-compose.yaml

# Star the container
docker compose up

# Conect database 
docker compose exec db bash
psql -U postgres -d products
\q
exit

# Shut down the conexion
docker compose down

SyntaxError: invalid syntax (1301526458.py, line 2)

`docker compose down` stops and removes the container and network, but leaves the volume intact. The next time you run docker compose up, your data will still be there. In order to remove all the information, such as the container information, the conexions and the volume, we need to run `docker compose down -v`. The `-v` is used to specifically remove the volume created using the docker image. 

The next time you run `docker compose uo`, your data will still be there, after running `docker compose down`.

### Questions
1. You run docker compose up and see a message like Error: bind: address already in use. What's the most likely cause?
   Port 5432 is already being used by another service on your machine. 
   This error typically means another process (like a local PostgreSQL installation) is already using port 5432. You can fix this by changing the host port mapping to something like 5433:5432 in your docker-compose.yaml file.

2. After running docker compose down, what happens to your database data stored in the pgdata volume?
   The data persists and will be available the next time you run docker compose up
   docker compose down stops and removes containers and networks, but it leaves volumes intact by default. This is a safety feature that prevents accidental data loss.

## Writing a Python ETL Script

To connect to a PostgreSQL database from Python, we’ll use a library called psycopg2. It’s a reliable, widely-used driver that lets our script execute SQL queries, manage transactions, and handle database errors.

1. Create a new file in the same folder called `app.py`. 
2. Import the libraries
3. Define the database connection settings. These will be read from environment variables that Docker Compose supplies when the script runs in a container.

To override the host when testing locally:
DB_HOST=localhost python app.py
DB_HOST=localhost DB_PORT=5433 python app.py

In [ ]:
# source python library
source /home/mponce/repos/.venv/bin/activate

# Install psycopg2 to connect a PostgreSQL database from Python
pip install psycopg2-binary

# Run script
python app.py

SyntaxError: invalid syntax (147114688.py, line 2)

### Questions

1. You're testing your app.py script locally and set DB_HOST=localhost when running it. Your docker-compose.yml maps the Postgres port as 5433:5432. Which port should you use in your local test?
   5433, because that's the host port exposed by Docker
   When connecting from your local machine (outside Docker), you use the host port (5433), which Docker maps to the container's internal port 5432.
2. You installed psycopg2-binary using pip, but your script imports psycopg2. What will happen when you run the script?
   The script will work correctly because psycopg2-binary installs the psycopg2 library
   The -binary suffix refers to the package distribution (with precompiled dependencies), but it installs the same psycopg2 library you import in your code.

## Containerizing the ETL Script and Running it with Postgres

1. Open the file `app.py`, we will keep working on it
2. Create a new row to add in the data table 
3. Connect to Postgres, insert the row and close the connection
4. Run the script

Here, we could have load the dataset from a CSV file or SQL but we will write a simple ETL operation that inserts a new row into vegetable table. If the vegetable table does not exist, the table must be created. This simulates a small “load” job like you might run on a schedule to append new data to a growing table.

Some considerations to take into account when adding a new line everytime we run the script. In order to avoid duplicate data, we should apply deduplication logic, such as:
1. Check for existing data before insert
2. using `ON CONFLICT` clauses
3. cleaning the table first with `TRUNCATE`

For the purpose of this project, we will not apply this steps.

In [ ]:
# Run docker container in the background
docker compose up -d

# Run python script 
DB_HOST=localhost python app.py

SyntaxError: invalid syntax (2002009610.py, line 2)

After running `python app.py`, I got the follwoing error: `Error during ETL: could not translate host name "db" to address: Temporary failure in name resolution`. The meaning of the error is that the script cannot find the database. The solution is to override the hostname when testing locally by using `DB_HOST=localhost python app.py`. The result expected in the command line is **ETL complete. 1 row inserted.**

You can verify the results by connecting to the database service and running a quick SQL query

In [ ]:
# Run container and PostgreSQL
docker compose exec db psql -U postgres -d products

SELECT * FROM vegetables;
# Output:
# id |   name   | form  | retail_price | cup_equivalent_price 
# ----+----------+-------+--------------+----------------------
#  1 | Parsnips | Fresh |         2.42 |                 2.19
# (1 row)

### Questions
1. You run python app.py multiple times and notice duplicate rows in your vegetables table. What's the most likely reason?
   The script inserts a new row every time it runs, regardless of whether identical data already exists

2. You're testing your ETL script locally and get the error: could not translate host name "db" to address. You need to connect to the Postgres container running on your machine. What should you change DB_HOST to?
   `localhost`
   When running the Python script directly on your host machine (not in a container), you need to use localhost or 127.0.0.1 to connect to the Postgres container. The hostname db only works for container-to-container communication within the Docker Compose network.

## Building a Custom Docker Image for the ETL App

At this point, I have created a python scripts that runs locally and connects to a containerized Postgres database. Now, I will containerize the script it self.

First, let's clarify some concepts:

- Docker image is the blueprint of the container. Defines wht the containers needs to works a such as the operating system, installed packages, environment variables, files, commands to run, ...  
- Docker container is the result of runnning the image to create a live isolated environment.

**Let's think about the image as the DNA which contains all the instructions for a cell to specialized and mantained it self. The container is the live cell running the processes.** 

What are the next steps to containerize the python script?
1. Create a dockerfile. It must be name Dockerfile.
2. Run the image
3. Run the container


### Dockerfile

Look for the file in the project folder. Let’s walk through what this file does:
- FROM python 3.10-slim: starts with a minimal Debian-based image that includes Python.\
- WORKDIR /app: creates a working directory where your code will live.
- COPY app.py . :copies your script into that directory inside the container.
- RUN pip install psycopg2-binary: installs the same Postgres driver you used locally.
- CMD [...]: sets the default command that will run when the container starts.

### Build the image

<docker buildx [OPTIONS] PATH | URL |>

- Path: Uses the current folder (.) as the build context
- "-t": Tags the resulting image with the name etl-app
- We do not use url
- Looks for a file called Dockerfile


In [ ]:
# Build the image
docker build -t etl-app .

# Check image was created
docker images

### Run the container

Now, it is time to run image using the name "etl-app". 

This will start the container and run the script, but unless your Postgres container is still running, it will likely fail with a connection error.

Right now, the Python container doesn’t know how to find the database because there’s no shared network, no environment variables, and no Compose setup. You’ll fix that in the next step by adding both services to a single Compose file.

In [ ]:
docker run etl-app
# Error during ETL: could not translate host name "db" to address: Temporary failure in name resolution

### Questions

1. You built a Docker image called etl-app and ran it with docker run etl-app, but it can't connect to your Postgres container that's already running. What's the most likely reason?
   The containers are not on the same Docker network by default.
   Containers started separately with docker run are isolated from each other. They need to be on the same network to communicate, which is why Docker Compose is useful—it creates a shared network automatically.
2. What happens if you modify app.py on your local machine after building the etl-app image?
   The running container will continue using the old version until you rebuild the image
   The COPY app.py . instruction copies the file into the image at build time. Changes to your local file won't affect containers created from that image until you rebuild it with docker build again.

## Updating Docker Compose for Multi-Service ETL Pipelines

Our Docker Compose is define and run by a single service which is the PostreSQL database. However, our aim is to run both the database and the app. For this reason, we will alter the `docker-compose.yaml` file to be able to run both services at once.

Docker Compose will handle building the app, starting both containers, connecting them over a shared network, and passing the right environment variables, all in one command. This setup makes it easy to swap out the app or run different versions just by updating the `docker-compose.yaml` file.

### Add app service to the compose file

We will tell Docker to:
- Build the app using the Dockerfile in your current folder
- Wait for the database to start before running
- Pass in environment variables so the app can connect to the Postgres container

We need add the following lines into the `docker-compose.yaml` file inside the services section, in order to perform the previous steps.

```
services: 
...

  app:
    build: .
    depends_on:
      - db
    environment:
      POSTGRES_USER: postgres
      POSTGRES_PASSWORD: postgres
      POSTGRES_DB: products
      DB_HOST: db
...

```

### Run and verify the full stack 

Both services are now define in a single file, and can be run using a single command `docker compose up --build -d`. This will rebuild our app image (if needed), launch both containers in the background, and connect them over a shared network. 

The image was rebuilt as `app-1`


In [ ]:
docker compose up --build -d
# => => exporting config sha256:925e6c62e9b6fba7f04a932dd74e88782150d28d4198153  0.0s
# => => exporting attestation manifest sha256:41a17100a5324775eebdc358d5d8ffeff  0.0s
# => => exporting manifest list sha256:9822a10c3364ae7256e1c0519d7a8f20d9544129  0.0s
# => => naming to docker.io/library/compose-demo-app:latest                      0.0s
# => => unpacking to docker.io/library/compose-demo-app:latest                   0.0s
# => resolving provenance for metadata file                                      0.0s
# [+] up 4/4
# ✔ Image compose-demo-app       Built                                            1.4s
# ✔ Network compose-demo_default Created                                          0.0s
# ✔ Container local_pg           Created                                          0.1s
# ✔ Container compose-demo-app-1 Created                                          0.0s

docker compose logs app
# app-1  | ETL complete. 1 row inserted.

# Play with the container/ Check the row was added
docker compose exec db psql -U postgres -d products

...

# Shut the container down
docker compose down

# Run the pipeline again
docker compose up 

Each time you restart the pipeline (`docker compose up`), the app container will re-run the script and insert the same row unless you update the script logic. In a real pipeline, you'd typically prevent that with checks, truncation, or ON CONFLICT clauses.

### Questions 
1. You run docker compose up and see a database connection error in your app logs, even though the db service is defined with depends_on. What is the most likely cause?
   depends_on only ensures the database container starts first, not that Postgres is ready to accept connections
   depends_on controls startup order but doesn't wait for the database to finish initializing. The app may try to connect before Postgres is ready. In production, you'd add retry logic or use a wait script.

2. After running docker compose down, you start the services again with docker compose up. What happens to the data that was previously inserted into the vegetables table?
   The data persists because the Compose file uses a named volume (pgdata) for the database
   Named volumes persist data between container lifecycles. docker compose down removes containers but keeps named volumes intact, so your database data survives restarts.

## Sharing Your Docker Image on Docker Hub
 
Now, we are only able to run our ETL app locally. In this step, we will be able a) to run our app in another machine, b) share it with our team, c) deploy it to the cloud or CI pipelines and d) backup version registry of your work.

In order to share our app, we need to use a container registry. A **container registry** is a storage for your containerized application images. A container registry, then, acts as both a collection of container repositories and a searchable catalogue where you manage and deploy images. Some examples of container registry available are:
- Docker Hub 
- Amazon ECR
- Harbor
- Azure Container Registry
- Github Container Registry
- Google Container Registry
- JFrog Container Registry
- Red Hat Quay

We will focus on [Docker Hub](hub.docker.com) which is the most common registry and offers free accounts and public repositories.

1. Create a Docker Hub account using the following [link](hub.docker.com).
2. Create a new repository, i.e. etl-app.
3. Tag your local image with your username and repository name.
   `docker tag <local-image> <username>/<repository>:<tag>`
   `docker tag etl-app username/etl-app:latest`
4. Login to docker
   `docker login`
5. Push the image 
   `docker push myname/etl-app:latest`
   
Now, you can pull and run the image from anywhere using `docker pull`

### Questions

1. You've built a Docker image called etl-app and want to push it to Docker Hub. Your Docker Hub username is datauser. Which command correctly tags the image for pushing?
   docker tag etl-app datauser/etl-app:latest
   The tag format is docker tag <local-image> <username>/<repository>:<tag>. This creates a reference to your image that Docker Hub will recognize.
2. After tagging your image as myname/etl-app:latest, you try to push it but get an authentication error. What's the most likely cause?
   You haven't run docker login to authenticate with Docker Hub
   Before pushing images to Docker Hub, you must authenticate using docker login. Docker needs your credentials to verify you have permission to push to that repository.